# Mask R-CNN + U-Net Hybrid -- Two-Stage Instance Mask Refinement

This notebook combines the two baselines already in this repo:

- **Stage 1 -- Mask R-CNN (ResNet-50 FPN)**, from
  `torchvision-s-mask-r-cnn-resnet-50-fpn-backbone.ipynb`: detects individual
  filament instances (boxes + a coarse 28x28 mask per box, upsampled to full
  resolution by torchvision internally).
- **Stage 2 -- a small U-Net refiner**, in the same style as the semantic
  U-Net in `solar-filament-segmentation-challenge-2026.ipynb`: takes the
  image crop + Stage 1's coarse mask for one detected instance and predicts a
  sharper, higher-resolution boundary for just that instance.

Why this combination: Mask R-CNN's own mask head operates at a fixed 28x28
resolution per RoI and is known to produce blobby boundaries -- the same
limitation this repo's mmdetection PointRend pipeline (see `README.md`) was
chosen to address. Rather than depending on mmdet/mmcv (which the README
documents as a painful install), this notebook gets a similar effect with
plain `torchvision` + `segmentation_models_pytorch`: Mask R-CNN still does the
hard part (separating touching/nearby filaments into distinct instances, which
a pure semantic U-Net can't do on its own -- that's why the other notebook
falls back to `cv2.connectedComponents`), and the U-Net only has to solve the
easier, narrower problem of boundary refinement inside one already-localized
crop.

**Known limitation of this design**: Stage 2 can only refine instances Stage 1
actually detected. A filament Mask R-CNN misses entirely stays missed -- this
notebook does not attempt to recover false negatives, only to sharpen true
positives' boundaries (see the validation section below for a Dice comparison
that quantifies the boundary-quality gain this actually buys).

In [ ]:
import os
import json
import warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.ops import box_iou
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from pycocotools.coco import COCO
import pycocotools.mask as mask_utils

try:
    import segmentation_models_pytorch as smp
except ImportError:
    os.system("pip install -q segmentation-models-pytorch")
    import segmentation_models_pytorch as smp

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

# Config

In [ ]:
CFG = {
    'base_dir': "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026",
    'val_fraction': 0.2,
    'seed': 42,

    # Stage 1 -- Mask R-CNN detector
    'det_epochs': 15,
    'det_lr': 0.005,
    'det_batch_size': 2,
    'det_score_thresh': 0.5,   # min detection confidence kept at inference/refiner-training time

    # Stage 2 -- mask refinement U-Net
    'roi_size': 256,           # fixed crop size fed to the refiner
    'roi_pad_ratio': 0.2,      # extra context padded around each box before cropping
    'match_iou_thresh': 0.5,   # min IoU for a Stage-1 prediction to count as "found" this GT instance
    'refine_epochs': 10,
    'refine_lr': 1e-3,
    'refine_batch_size': 16,
    'refine_backbone': 'resnet18',
}

BASE_DIR = Path(CFG['base_dir'])
TRAIN_IMG_DIR = BASE_DIR / "train/train_images"
TEST_IMG_DIR = BASE_DIR / "test/test_images"
TRAIN_JSON_PATH = BASE_DIR / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# Dataset & Preprocessing (Stage 1)

Reused from `torchvision-s-mask-r-cnn-resnet-50-fpn-backbone.ipynb`, with one
correctness fix: the box-clamp now uses each image's *actual* height/width
(from the COCO JSON) instead of a hardcoded `2048`.

In [ ]:
def remove_limb_darkening(image_gray, kernel_size=151):
    """
    Equalizes solar background brightness by dividing by a large median filter.
    """
    bg = cv2.medianBlur(image_gray, kernel_size)
    bg = np.maximum(bg, 1)
    corrected = (image_gray.astype(np.float32) / bg.astype(np.float32)) * 128.0
    return np.clip(corrected, 0, 255).astype(np.uint8)


class SolarFilamentDataset(Dataset):
    """
    Parses COCO JSON, applies limb-darkening correction, and outputs Mask
    R-CNN targets (boxes/labels/masks at full image resolution).
    """
    def __init__(self, image_dir, json_path):
        self.image_dir = image_dir
        self.coco = COCO(json_path)
        self.img_ids = list(self.coco.imgs.keys())

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.coco.imgs[img_id]
        h, w = img_info['height'], img_info['width']

        img_path = os.path.join(self.image_dir, img_info['file_name'])
        image_gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if image_gray is None:
            raise FileNotFoundError(f"Image not found at {img_path}")

        image_corr = remove_limb_darkening(image_gray)
        image_rgb = np.stack([image_corr] * 3, axis=-1).astype(np.float32) / 255.0
        image_norm = (image_rgb - MEAN) / STD
        image_tensor = torch.from_numpy(image_norm).permute(2, 0, 1).float()

        ann_ids = self.coco.getAnnIds(imgIds=[img_id])
        anns = self.coco.loadAnns(ann_ids)

        masks, boxes, labels = [], [], []
        for ann in anns:
            mask = self.coco.annToMask(ann)
            if mask.sum() == 0:
                continue
            x, y, bw, bh = ann['bbox']
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(w, x + bw), min(h, y + bh)
            if x2 <= x1 or y2 <= y1:
                continue
            masks.append(mask)
            boxes.append([x1, y1, x2, y2])
            labels.append(1)  # binary class: 1 = filament

        if len(masks) > 0:
            target = {
                "boxes": torch.as_tensor(boxes, dtype=torch.float32),
                "labels": torch.as_tensor(labels, dtype=torch.int64),
                "masks": torch.as_tensor(np.stack(masks, axis=0), dtype=torch.uint8),
                "image_id": torch.tensor([idx]),
            }
        else:
            target = {
                "boxes": torch.zeros((0, 4), dtype=torch.float32),
                "labels": torch.zeros((0,), dtype=torch.int64),
                "masks": torch.zeros((0, h, w), dtype=torch.uint8),
                "image_id": torch.tensor([idx]),
            }

        return image_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))

# Stage 1 Model -- Mask R-CNN (ResNet-50 FPN)

Reused as-is from the torchvision notebook.

In [ ]:
def get_maskrcnn_model(num_classes=2):
    """
    Mask R-CNN with ResNet-50 FPN backbone.
    Sets zero mean/std to skip internal CUDA normalization kernels (we
    already normalize ourselves in the dataset).
    """
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(
        weights="DEFAULT",
        image_mean=[0.0, 0.0, 0.0],
        image_std=[1.0, 1.0, 1.0]
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    return model


def compute_dice_score(pred_masks, gt_masks, score_thresh=0.5):
    """
    Bipartite-matched mean Dice score between predicted and GT masks.
    """
    if len(pred_masks) == 0 and len(gt_masks) == 0:
        return 1.0
    if len(pred_masks) == 0 or len(gt_masks) == 0:
        return 0.0

    pred_binary = (pred_masks > score_thresh).astype(np.float32)
    gt_binary = (gt_masks > 0).astype(np.float32)

    pred_flat = pred_binary.reshape(pred_binary.shape[0], -1)
    gt_flat = gt_binary.reshape(gt_binary.shape[0], -1)

    intersection = np.dot(pred_flat, gt_flat.T)
    pred_sums = pred_flat.sum(axis=1, keepdims=True)
    gt_sums = gt_flat.sum(axis=1, keepdims=True)

    dice_matrix = (2.0 * intersection) / (pred_sums + gt_sums.T + 1e-6)

    matched_dice = []
    while dice_matrix.size > 0 and dice_matrix.max() > 0:
        max_idx = np.unravel_index(np.argmax(dice_matrix), dice_matrix.shape)
        matched_dice.append(dice_matrix[max_idx])
        dice_matrix = np.delete(dice_matrix, max_idx[0], axis=0)
        dice_matrix = np.delete(dice_matrix, max_idx[1], axis=1)

    return np.mean(matched_dice) if len(matched_dice) > 0 else 0.0

# Stage 1 Data Loaders & Training Setup

In [ ]:
full_dataset = SolarFilamentDataset(image_dir=TRAIN_IMG_DIR, json_path=TRAIN_JSON_PATH)

val_size = int(CFG['val_fraction'] * len(full_dataset))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(CFG['seed'])
)

train_loader = DataLoader(train_dataset, batch_size=CFG['det_batch_size'], shuffle=True,
                           num_workers=2, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False,
                         num_workers=2, collate_fn=collate_fn)

print(f"Dataset loaded: Train = {len(train_dataset)} | Val = {len(val_dataset)}")

detector = get_maskrcnn_model(num_classes=2).to(device)

det_params = [p for p in detector.parameters() if p.requires_grad]
det_optimizer = torch.optim.SGD(det_params, lr=CFG['det_lr'], momentum=0.9, weight_decay=0.0005)
det_scheduler = torch.optim.lr_scheduler.StepLR(det_optimizer, step_size=3, gamma=0.1)

det_history = {"train_loss": [], "val_loss": [], "val_dice": []}

# Stage 1 Training Loop -- Mask R-CNN

In [ ]:
for epoch in range(CFG['det_epochs']):
    detector.train()
    running_train_loss = 0.0
    pbar = tqdm(train_loader, desc=f"[Detector] Epoch {epoch+1}/{CFG['det_epochs']} [Train]")
    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = detector(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        det_optimizer.zero_grad()
        losses.backward()
        det_optimizer.step()

        running_train_loss += losses.item()
        pbar.set_postfix({"loss": f"{losses.item():.4f}"})

    avg_train_loss = running_train_loss / len(train_loader)
    det_scheduler.step()

    # torchvision detection models only return losses in train() mode
    detector.train()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            val_loss_dict = detector(images, targets)
            running_val_loss += sum(loss for loss in val_loss_dict.values()).item()
    avg_val_loss = running_val_loss / len(val_loader)

    detector.eval()
    val_dice_scores = []
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            preds = detector(images)
            p_masks = preds[0]['masks'].squeeze(1).detach().cpu().numpy()
            g_masks = targets[0]['masks'].detach().cpu().numpy()
            val_dice_scores.append(compute_dice_score(p_masks, g_masks, score_thresh=CFG['det_score_thresh']))
    avg_val_dice = float(np.mean(val_dice_scores))

    det_history["train_loss"].append(avg_train_loss)
    det_history["val_loss"].append(avg_val_loss)
    det_history["val_dice"].append(avg_val_dice)

    print(f"[Detector] Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | Val Dice (raw Mask R-CNN): {avg_val_dice:.4f}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

torch.save(detector.state_dict(), "maskrcnn_stage1.pth")
print("Stage 1 detector saved to maskrcnn_stage1.pth")

# Stage 2 -- RoI Crop Extraction for Mask Refinement

Runs the trained Stage-1 detector once over the *same* train/val split, greedily
matches each prediction to a GT instance by box IoU, and for every matched pair
crops out (image, Stage-1 coarse mask, GT mask) around that instance's box.
These crops become the refiner's training data.

Predictions with no matching GT (false positives) and GT instances with no
matching prediction (missed detections) are dropped here -- there's no mask to
refine or supervise against in either case.

In [ ]:
def match_predictions_to_gt(pred_boxes, gt_boxes, iou_thresh):
    """
    Greedy one-to-one matching of predicted boxes to GT boxes by IoU.
    Returns a list of (pred_idx, gt_idx) pairs, each above iou_thresh.
    """
    if len(pred_boxes) == 0 or len(gt_boxes) == 0:
        return []

    ious = box_iou(torch.as_tensor(pred_boxes), torch.as_tensor(gt_boxes)).numpy()
    flat = [(ious[i, j], i, j) for i in range(ious.shape[0]) for j in range(ious.shape[1])]
    flat.sort(key=lambda t: t[0], reverse=True)

    matches, used_pred, used_gt = [], set(), set()
    for iou, i, j in flat:
        if iou < iou_thresh:
            break
        if i in used_pred or j in used_gt:
            continue
        matches.append((i, j))
        used_pred.add(i)
        used_gt.add(j)
    return matches


def crop_and_resize(array, box, pad_ratio, out_size, is_mask):
    """
    Crops `array` (H,W) or (H,W,C) around `box` = [x1,y1,x2,y2] with extra
    context padding, then resizes to (out_size, out_size).
    """
    h, w = array.shape[:2]
    x1, y1, x2, y2 = box
    pad_x, pad_y = (x2 - x1) * pad_ratio, (y2 - y1) * pad_ratio

    cx1 = int(max(0, x1 - pad_x))
    cy1 = int(max(0, y1 - pad_y))
    cx2 = int(min(w, x2 + pad_x))
    cy2 = int(min(h, y2 + pad_y))
    if cx2 <= cx1 or cy2 <= cy1:
        return None, None

    crop = array[cy1:cy2, cx1:cx2]
    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    resized = cv2.resize(crop, (out_size, out_size), interpolation=interp)
    return resized, (cx1, cy1, cx2, cy2)


def paste_mask(mask_roi, roi_box, canvas_shape):
    """
    Resizes `mask_roi` back to `roi_box`'s pixel extent and pastes it into a
    full-resolution zero canvas.
    """
    cx1, cy1, cx2, cy2 = roi_box
    canvas = np.zeros(canvas_shape, dtype=np.float32)
    resized = cv2.resize(mask_roi, (cx2 - cx1, cy2 - cy1), interpolation=cv2.INTER_LINEAR)
    canvas[cy1:cy2, cx1:cx2] = resized
    return canvas


def build_refinement_samples(detector, dataset, device, cfg):
    """
    Returns a list of (input_4ch, gt_mask) crop pairs -- one per Stage-1
    prediction that matched a GT instance -- to supervise the Stage-2 refiner.
    """
    detector.eval()
    samples = []
    loader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Extracting RoI crops"):
            image_t = images[0]
            target = targets[0]

            pred = detector([image_t.to(device)])[0]
            keep = pred['scores'].cpu().numpy() > cfg['det_score_thresh']
            pred_boxes = pred['boxes'].cpu().numpy()[keep]
            pred_masks = pred['masks'].squeeze(1).cpu().numpy()[keep]

            gt_boxes = target['boxes'].numpy()
            gt_masks = target['masks'].numpy()
            if len(gt_boxes) == 0 or len(pred_boxes) == 0:
                continue

            matches = match_predictions_to_gt(pred_boxes, gt_boxes, cfg['match_iou_thresh'])
            if not matches:
                continue

            image_rgb = np.clip(image_t.permute(1, 2, 0).numpy() * STD + MEAN, 0, 1)

            for pred_idx, gt_idx in matches:
                box = pred_boxes[pred_idx]
                img_crop, roi_box = crop_and_resize(image_rgb, box, cfg['roi_pad_ratio'], cfg['roi_size'], is_mask=False)
                if img_crop is None:
                    continue
                img_crop = (img_crop - MEAN) / STD
                coarse_crop, _ = crop_and_resize(pred_masks[pred_idx], box, cfg['roi_pad_ratio'], cfg['roi_size'], is_mask=False)
                gt_crop, _ = crop_and_resize(gt_masks[gt_idx].astype(np.float32), box, cfg['roi_pad_ratio'], cfg['roi_size'], is_mask=True)

                input_4ch = np.concatenate([img_crop, coarse_crop[..., None]], axis=-1)  # (roi, roi, 4)
                samples.append((
                    torch.from_numpy(input_4ch).permute(2, 0, 1).float(),
                    torch.from_numpy(gt_crop).unsqueeze(0).float(),
                ))

    return samples


class RefinementDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [ ]:
train_refine_samples = build_refinement_samples(detector, train_dataset, device, CFG)
val_refine_samples = build_refinement_samples(detector, val_dataset, device, CFG)
print(f"Refiner training crops: {len(train_refine_samples)} | val crops: {len(val_refine_samples)}")

train_refine_loader = DataLoader(RefinementDataset(train_refine_samples), batch_size=CFG['refine_batch_size'], shuffle=True)
val_refine_loader = DataLoader(RefinementDataset(val_refine_samples), batch_size=CFG['refine_batch_size'], shuffle=False)

# Stage 2 Model -- Mask Refinement U-Net

Same `segmentation_models_pytorch` U-Net + Dice/BCE loss as
`solar-filament-segmentation-challenge-2026.ipynb`, but with `in_channels=4`
(RGB crop + Stage-1 coarse mask) and a lighter `resnet18` encoder since each
input is a small, already-localized crop rather than a full image.

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True)

    def forward(self, y_pred, y_true):
        return 0.5 * self.bce(y_pred, y_true) + 0.5 * self.dice(y_pred, y_true)


def build_refiner():
    return smp.Unet(
        encoder_name=CFG['refine_backbone'],
        encoder_weights='imagenet',
        in_channels=4,
        classes=1,
        activation=None,  # output logits
    )


refiner = build_refiner().to(device)
refine_criterion = DiceBCELoss()
refine_optimizer = torch.optim.AdamW(refiner.parameters(), lr=CFG['refine_lr'])
refine_history = {"train_loss": [], "val_loss": []}

# Stage 2 Training Loop -- Mask Refinement U-Net

In [ ]:
best_refine_val_loss = float('inf')

for epoch in range(CFG['refine_epochs']):
    refiner.train()
    train_loss = 0.0
    for x, y in tqdm(train_refine_loader, desc=f"[Refiner] Epoch {epoch+1}/{CFG['refine_epochs']} [Train]"):
        x, y = x.to(device), y.to(device)
        refine_optimizer.zero_grad()
        out = refiner(x)
        loss = refine_criterion(out, y)
        loss.backward()
        refine_optimizer.step()
        train_loss += loss.item()
    avg_train_loss = train_loss / len(train_refine_loader)

    refiner.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_refine_loader:
            x, y = x.to(device), y.to(device)
            out = refiner(x)
            val_loss += refine_criterion(out, y).item()
    avg_val_loss = val_loss / len(val_refine_loader)

    refine_history["train_loss"].append(avg_train_loss)
    refine_history["val_loss"].append(avg_val_loss)
    print(f"[Refiner] Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_refine_val_loss:
        best_refine_val_loss = avg_val_loss
        torch.save(refiner.state_dict(), "best_refiner.pth")
        print("Best refiner saved!")

# Combined Hybrid Inference Pipeline

`hybrid_predict` runs Stage 1 to get instance boxes + coarse masks, then runs
each detected instance through Stage 2 for a refined boundary, and pastes the
result back into a full-resolution canvas.

In [ ]:
refiner.load_state_dict(torch.load("best_refiner.pth", map_location=device))
refiner.eval()


def hybrid_predict(image_tensor, detector, refiner, device, cfg, mask_thresh=0.5):
    """
    Stage 1 (Mask R-CNN) proposes & separates instances; Stage 2 (U-Net)
    refines each instance's mask boundary within its own RoI crop.
    Returns a list of full-resolution binary instance masks.
    """
    h, w = image_tensor.shape[1:]
    image_rgb = np.clip(image_tensor.permute(1, 2, 0).numpy() * STD + MEAN, 0, 1)

    with torch.no_grad():
        pred = detector([image_tensor.to(device)])[0]

    keep = pred['scores'].cpu().numpy() > cfg['det_score_thresh']
    boxes = pred['boxes'].cpu().numpy()[keep]
    coarse_masks = pred['masks'].squeeze(1).cpu().numpy()[keep]

    instance_masks = []
    for box, coarse_mask in zip(boxes, coarse_masks):
        img_crop, roi_box = crop_and_resize(image_rgb, box, cfg['roi_pad_ratio'], cfg['roi_size'], is_mask=False)
        if img_crop is None:
            continue
        img_crop = (img_crop - MEAN) / STD
        coarse_crop, _ = crop_and_resize(coarse_mask, box, cfg['roi_pad_ratio'], cfg['roi_size'], is_mask=False)

        input_4ch = np.concatenate([img_crop, coarse_crop[..., None]], axis=-1)
        input_t = torch.from_numpy(input_4ch).permute(2, 0, 1).float().unsqueeze(0).to(device)

        with torch.no_grad():
            refined_prob = torch.sigmoid(refiner(input_t))[0, 0].cpu().numpy()

        full_mask = paste_mask(refined_prob, roi_box, (h, w))
        instance_masks.append((full_mask > mask_thresh).astype(np.uint8))

    return instance_masks

# Validation -- Hybrid vs. Raw Mask R-CNN (Dice Comparison)

Quantifies what Stage 2 actually buys over Stage 1 alone, on the same held-out
validation split.

In [ ]:
raw_dice_scores, hybrid_dice_scores = [], []
val_loader_single = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

detector.eval()
with torch.no_grad():
    for images, targets in tqdm(val_loader_single, desc="Comparing Raw vs Hybrid"):
        image_t = images[0]
        target = targets[0]
        gt_masks = target['masks'].numpy()
        if len(gt_masks) == 0:
            continue

        pred = detector([image_t.to(device)])[0]
        p_masks = pred['masks'].squeeze(1).cpu().numpy()
        raw_dice_scores.append(compute_dice_score(p_masks, gt_masks, score_thresh=CFG['det_score_thresh']))

        hybrid_masks = hybrid_predict(image_t, detector, refiner, device, CFG)
        if len(hybrid_masks) == 0:
            hybrid_dice_scores.append(0.0)
            continue
        hybrid_stack = np.stack(hybrid_masks, axis=0).astype(np.float32)
        hybrid_dice_scores.append(compute_dice_score(hybrid_stack, gt_masks, score_thresh=0.5))

print(f"Raw Mask R-CNN Val Dice:             {np.mean(raw_dice_scores):.4f}")
print(f"Hybrid (Mask R-CNN + U-Net) Val Dice: {np.mean(hybrid_dice_scores):.4f}")

# Test Inference & Submission (COCO RLE)

In [ ]:
def load_test_image_tensor(img_path):
    image_gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    image_corr = remove_limb_darkening(image_gray)
    image_rgb = np.stack([image_corr] * 3, axis=-1).astype(np.float32) / 255.0
    image_norm = (image_rgb - MEAN) / STD
    return torch.from_numpy(image_norm).permute(2, 0, 1).float()


def mask_to_coco_rle(binary_mask):
    """
    Encodes a binary numpy mask (H, W) into COCO compressed RLE ASCII string.
    """
    fortran_mask = np.asfortranarray(binary_mask.astype(np.uint8))
    rle = mask_utils.encode(fortran_mask)
    rle['counts'] = rle['counts'].decode('utf-8')
    return rle['counts']


test_files = sorted(TEST_IMG_DIR.glob("*.jpeg"))
submissions = []

detector.eval()
refiner.eval()

print("Generating Hybrid (Mask R-CNN + U-Net) Predictions for Test Images...")

for img_path in tqdm(test_files, desc="Hybrid inference on test set"):
    image_t = load_test_image_tensor(img_path)
    instance_masks = hybrid_predict(image_t, detector, refiner, device, CFG)

    image_name = img_path.stem
    filament_count = 0
    for mask in instance_masks:
        if mask.sum() < 30:  # filter out tiny noise artifacts
            continue
        filament_count += 1
        submissions.append({
            'filament_id': f"{image_name}_{filament_count}",
            'segmentation_rle': mask_to_coco_rle(mask),
        })

    # Fallback: if no filaments detected, encode an empty background mask
    if filament_count == 0:
        empty_mask = np.zeros((image_t.shape[1], image_t.shape[2]), dtype=np.uint8)
        submissions.append({
            'filament_id': f"{image_name}_1",
            'segmentation_rle': mask_to_coco_rle(empty_mask),
        })

sub_df = pd.DataFrame(submissions)

print("\nVerification check for NULL/NaN values:")
print(sub_df.isnull().sum())
print(f"\nTotal predictions generated: {len(sub_df)}")

sub_df.to_csv("submission.csv", index=False)
print("submission.csv generated successfully with COCO Compressed RLE strings!")
sub_df.head(10)

# Notes & Limitations

- **Missed detections aren't recovered.** Stage 2 only refines instances
  Stage 1 already found (see the caveat at the top). If Mask R-CNN's recall is
  the bottleneck rather than boundary precision, this hybrid won't help --
  check the Dice comparison above before investing further here.
- **`roi_size=256` is fixed.** If filaments are frequently much larger than
  256px (long thin structures spanning a big fraction of the 2048x2048 frame
  are plausible for this dataset), the crop+resize step in `crop_and_resize`
  may down-sample large instances more than intended. Check the distribution
  of `(x2-x1, y2-y1)` box sizes in the training set and raise `roi_size` (or
  make it adaptive per-box) if so.
- **Greedy IoU matching** (`match_predictions_to_gt`) is not globally optimal
  (unlike a Hungarian assignment) but is standard practice for this kind of
  detector-output-to-GT matching and is cheap.
- **Two training phases run sequentially, not jointly.** Stage 1 is fully
  trained and frozen before Stage 2 ever sees data. An end-to-end joint
  fine-tune (backprop refiner loss into the detector's mask head too) is a
  natural next step if this two-phase version shows a clear Dice gain.